# 12.8 PyG classification and explanations: an audited BBBP example

**Research scenario:** a team has a historical table of binary permeability labels and wants to rank molecules for follow-up. A useful analysis must address label quality, chemical novelty, class imbalance, errors and explanation limits. This notebook trains a real PyG graph classifier and then uses **PyG's `Explainer` and `GNNExplainer`** to inspect one model prediction.

The bundled BBBP dataset records a heterogeneous binary blood–brain-barrier permeability label. We predict that benchmark label only; the result is not a validated clinical probability, a complete pharmacokinetic model, or a recommendation to administer a compound. [Dataset provenance](datasets/README.md) and [MoleculeNet](https://doi.org/10.1039/C7SC02664A).

## What to learn

1. Curate invalid, repeated and conflicting records while preserving row identities.
2. Train a graph-level binary classifier using logits and a loss with training-derived class weights.
3. Select a decision threshold on validation data and report complementary test metrics.
4. Run a real graph explainer, inspect its masks and test what those masks do.

**Prerequisites:** [12.6 PyG basics](Chapter12_Part6.ipynb), [12.7 training](Chapter12_Part7.ipynb), and [12.4 explanation concepts](Chapter12_Part4.ipynb). **Runtime:** at most 500 molecules, 50 small CPU epochs and two 25-step explanation fits. All data are local; no preceding notebook output or downloaded model is required.

Read the classifier as the core lesson; explanation masks are a deeper diagnostic exercise. A highlighted substructure explains a model calculation only under a stated method and contrast.

For a binary model explanation, PyG infers the target class at raw logit zero (sigmoid score 0.5). This is its explanation convention. If a screening threshold differs, distinguish the class being explained from the downstream screening decision.

In [ ]:
import os
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[key] = '1'
from pathlib import Path
import hashlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
import torch
from torch import nn
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, GCNConv, GINEConv, global_mean_pool
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Scaffolds import MurckoScaffold

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
SEED = 2026
torch.manual_seed(SEED)
OUT = Path('outputs/chapter12_part8')
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {'torch': str(torch.__version__), 'torch_geometric': torch_geometric.__version__,
            'rdkit': rdBase.rdkitVersion, 'numpy': np.__version__}
print(VERSIONS)
print('CPU only; no compiled graph extensions or dataset downloads.')

## 12.8.1 Audit before selecting a teaching subset

`p_np=1` is the reported positive class and `p_np=0` the negative class. The CSV includes invalid SMILES, repeated structures and structures with conflicting labels. We exclude invalid records, disconnected inputs and **all records in conflicting canonical-label groups**, retain one row from each remaining canonical structure, then use a label-independent hash to select at most 500 connected molecules with at most 60 heavy atoms. This policy changes the population; it does not establish that every excluded measurement is wrong.

Source-row IDs follow every retained graph. Canonicalization here preserves isomeric SMILES, without tautomer, protonation or salt standardization. Missing experimental context cannot be repaired by neural-network training.

In [ ]:
data_path = Path('datasets/BBBP.csv')
source_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert source_sha256 == 'd07a38487aeac5cee5508413e468043ef3097451d2a112701c2d60be9ec6b662'
raw = pd.read_csv(data_path)
assert raw.p_np.isin([0,1]).all()
rows = []
with rdBase.BlockLogs():
    for row_id, row in raw.iterrows():
        mol = Chem.MolFromSmiles(row.smiles) if isinstance(row.smiles,str) else None
        valid = mol is not None and mol.GetNumAtoms() > 0
        rows.append({'source_row': int(row_id), 'smiles': row.smiles, 'y': int(row.p_np), 'mol': mol,
            'valid': valid, 'canonical': Chem.MolToSmiles(mol,isomericSmiles=True) if valid else None,
            'connected': valid and len(Chem.GetMolFrags(mol)) == 1,
            'heavy_atoms': mol.GetNumHeavyAtoms() if valid else 0})
audit = pd.DataFrame(rows)
label_counts = audit.loc[audit.valid].groupby('canonical').y.nunique()
conflicts = set(label_counts[label_counts > 1].index)
audit['conflict'] = audit.canonical.isin(conflicts)
eligible = audit.valid & audit.connected & ~audit.conflict & audit.heavy_atoms.between(1,60)
unique = audit.loc[eligible].drop_duplicates('canonical').copy()
unique['selection_key'] = unique.canonical.map(lambda s: hashlib.sha256(s.encode()).hexdigest())
sample = unique.sort_values(['selection_key','source_row']).head(500).copy().reset_index(drop=True)
audit['selected'] = audit.source_row.isin(sample.source_row)
audit.drop(columns='mol').to_csv(OUT / 'source_audit.csv', index=False)
print({'source_rows':len(raw), 'invalid_SMILES':int((~audit.valid).sum()),
       'conflicting_canonical_groups':len(conflicts), 'unique_eligible':len(unique), 'selected':len(sample)})
molecules = sample.mol.tolist()
sample['group'] = [MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False) or 'ACYCLIC' for m in molecules]

## 12.8.2 Split scaffold groups and inspect class support

First hold out test scaffold groups; then split the remaining groups into training and validation. The seeds and group fractions are fixed before training. Each partition must contain both classes for the metrics used here. Group fractions are not guaranteed row fractions, especially for the shared acyclic group.

The graph representation omits stereo, so non-stereochemical scaffold grouping also helps keep closely corresponding inputs together. Exact distinct scaffolds can still be similar. These are small teaching partitions, not official BBBP benchmark splits.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
outer = GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=SEED)
development_id, test_id = next(outer.split(sample, groups=sample.group))
inner = GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=SEED+1)
train_local, val_local = next(inner.split(sample.iloc[development_id],groups=sample.iloc[development_id].group))
train_id, val_id = development_id[train_local], development_id[val_local]
indices = {'train':train_id,'validation':val_id,'test':test_id}
sample['partition'] = ''
for label, ids in indices.items(): sample.loc[ids,'partition'] = label
assert sample.groupby('canonical').partition.nunique().max() == 1
assert sample.groupby('group').partition.nunique().max() == 1
for ids in indices.values(): assert set(sample.iloc[ids].y) == {0,1}
support = pd.crosstab(sample.partition,sample.y).reindex(['train','validation','test'])
display(support.rename(columns={0:'negative',1:'positive'}))
fig, ax = plt.subplots(figsize=(6.5,3.3),layout='constrained')
ax.bar(support.index,support[0],label='Negative label',color='#337f9b')
ax.bar(support.index,support[1],bottom=support[0],label='Positive label',color='#d39232')
ax.set(ylabel='Molecules',title='Class support in each held-out chemical partition')
ax.legend(fontsize=8)
fig.savefig(OUT/'class_support.png',dpi=140)
plt.show()
sample.drop(columns=['mol','selection_key']).to_csv(OUT/'split.csv',index=False)

## 12.8.3 A graph classifier outputs a logit

We reuse the compact 17-atom-feature, 7-bond-feature encoder and two-layer `GINEConv` model. Its output $z$ is a **logit**, an unrestricted number. The sigmoid $\sigma(z)=1/(1+e^{-z})$ converts it to a score between zero and one. Pass raw logits to `BCEWithLogitsLoss`; applying sigmoid before that loss is incorrect.

Class weighting changes the objective: `pos_weight = training negatives / training positives` balances the total class contributions in the weighted binary loss. It is estimated from training labels only. Weighting means the resulting sigmoid score should **not automatically be interpreted as a calibrated probability in the original population**. We use validation data to choose a threshold and retain the raw scores for ranking assessment.

In [ ]:
ELEMENTS = [5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
NODE_NAMES = [f'element_{z}' for z in ELEMENTS] + ['element_other', 'degree/4',
              'attached_H/4', 'formal_charge/2', 'aromatic', 'in_ring']
EDGE_NAMES = ['single', 'double', 'triple', 'aromatic', 'other', 'conjugated', 'in_ring']
FEATURE_SCHEMA = {'elements': ELEMENTS, 'node_names': NODE_NAMES, 'edge_names': EDGE_NAMES,
    'hydrogens': 'RDKit default implicit H; attached counts on atoms',
    'components': 'connected only', 'stereochemistry': 'omitted', 'coordinates': 'omitted'}

def one_hot_other(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]

def mol_to_data(mol, target=None, source_row=None):
    if mol is None or mol.GetNumAtoms() == 0 or len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError('Provide a nonempty connected molecule.')
    nodes = [one_hot_other(a.GetAtomicNum(), ELEMENTS) +
             [a.GetDegree()/4, a.GetTotalNumHs()/4, a.GetFormalCharge()/2,
              float(a.GetIsAromatic()), float(a.IsInRing())] for a in mol.GetAtoms()]
    edges, attributes = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        attr = one_hot_other(b.GetBondType(), BOND_TYPES) + [float(b.GetIsConjugated()), float(b.IsInRing())]
        edges.extend([(i, j), (j, i)])
        attributes.extend([attr, attr])
    data = Data(x=torch.tensor(nodes, dtype=torch.float32),
        edge_index=torch.tensor(edges, dtype=torch.long).reshape(-1, 2).T.contiguous(),
        edge_attr=torch.tensor(attributes, dtype=torch.float32).reshape(-1, len(EDGE_NAMES)),
        num_nodes=mol.GetNumAtoms())
    if target is not None:
        data.y = torch.tensor([target], dtype=torch.float32)
    if source_row is not None:
        data.source_row = torch.tensor([source_row], dtype=torch.long)
    data.validate(raise_on_error=True)
    return data

In [ ]:
class PropertyGINE(nn.Module):
    def __init__(self, node_dim=17, edge_dim=7, hidden=24, layers=2):
        super().__init__()
        self.encoder = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GINEConv(
            nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)),
            edge_dim=edge_dim, train_eps=False) for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden+1, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x, edge_index, edge_attr, batch):
        h = torch.relu(self.encoder(x))
        for conv in self.convs:
            h = h + torch.relu(conv(h, edge_index, edge_attr))
        pooled = global_mean_pool(h, batch)
        counts = torch.bincount(batch, minlength=len(pooled)).to(h.dtype).unsqueeze(1)
        return self.head(torch.cat([pooled, torch.log1p(counts)], dim=1)).squeeze(1)

def forward_batch(model, data):
    return model(data.x, data.edge_index, data.edge_attr, data.batch)

ARCHITECTURE = {'node_dim': len(NODE_NAMES), 'edge_dim': len(EDGE_NAMES), 'hidden': 24, 'layers': 2}

In [ ]:
y = sample.y.to_numpy(dtype=int)
graphs = [mol_to_data(m, target=int(y[i]), source_row=int(sample.iloc[i].source_row)) for i,m in enumerate(molecules)]
datasets = {p:[graphs[i] for i in ids] for p,ids in indices.items()}
full_batches = {p:Batch.from_data_list(items) for p,items in datasets.items()}
train_loader = DataLoader(datasets['train'],batch_size=64,shuffle=True,num_workers=0,
                         generator=torch.Generator().manual_seed(SEED))
positive_weight = float(np.sum(y[train_id] == 0)/np.sum(y[train_id] == 1))
loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight))
torch.manual_seed(SEED)
model = PropertyGINE(**ARCHITECTURE)
optimizer = torch.optim.Adam(model.parameters(),lr=0.005,weight_decay=1e-4)
MAX_EPOCHS, PATIENCE, MIN_DELTA = 50, 8, 1e-4
print('Training positive fraction:', y[train_id].mean(), '| pos_weight:',positive_weight)
print('Graph output / target:',forward_batch(model,full_batches['train']).shape,full_batches['train'].y.shape)

## 12.8.4 Select the model with validation loss

The fixed budget is 50 epochs with patience 8. Validation uses the same training-derived loss weights, so its loss measures the same objective. We restore the actual minimum-validation-loss weights. Neither the classifier settings nor the stopping decision uses test labels.

In [ ]:
best_loss, patience_reference = float('inf'),float('inf')
best_state, best_epoch, stale = None,None,0
history_rows=[]
start=time.perf_counter()
for epoch in range(1,MAX_EPOCHS+1):
    model.train()
    for mini_batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss=loss_function(forward_batch(model,mini_batch),mini_batch.y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.inference_mode():
        train_loss=loss_function(forward_batch(model,full_batches['train']),full_batches['train'].y).item()
        val_loss=loss_function(forward_batch(model,full_batches['validation']),full_batches['validation'].y).item()
    assert np.isfinite([train_loss,val_loss]).all()
    history_rows.append({'epoch':epoch,'training_weighted_BCE':train_loss,'validation_weighted_BCE':val_loss})
    if val_loss<best_loss:
        best_loss,best_epoch=val_loss,epoch
        best_state={k:v.detach().clone() for k,v in model.state_dict().items()}
    if val_loss<patience_reference-MIN_DELTA:
        patience_reference,stale=val_loss,0
    else: stale+=1
    if stale>=PATIENCE: break
model.load_state_dict(best_state)
model.eval()
training_seconds=time.perf_counter()-start
with torch.inference_mode():
    assert np.isclose(loss_function(forward_batch(model,full_batches['validation']),full_batches['validation'].y).item(),best_loss)
history=pd.DataFrame(history_rows)
history.to_csv(OUT/'learning_history.csv',index=False)
fig,ax=plt.subplots(figsize=(6.7,3.3),layout='constrained')
ax.plot(history.epoch,history.training_weighted_BCE,label='Training')
ax.plot(history.epoch,history.validation_weighted_BCE,label='Validation')
ax.axvline(best_epoch,color='black',linestyle=':',label=f'Restored epoch {best_epoch}')
ax.set(xlabel='Epoch',ylabel='Weighted binary cross-entropy',title='Select weights without test labels')
ax.legend(fontsize=8)
fig.savefig(OUT/'learning_curves.png',dpi=140)
plt.show()
print(f'{epoch} epochs in {training_seconds:.2f} s; restored epoch {best_epoch}.')

## 12.8.5 Thresholds answer a decision question

A high score may rank a molecule above another without telling us where to draw a binary decision boundary. We predeclare five candidate thresholds and select the one with highest **validation balanced accuracy**, the mean of sensitivity and specificity. Ties choose the lower threshold. In a real project, false-positive/false-negative costs or experimental capacity may imply a different selection criterion.

For context we fit a class-weighted logistic regression on radius-2, 1,024-bit Morgan fingerprints using the same training molecules. It gets the same threshold-selection rule. No parameter search is performed. The test remains reserved for the frozen model/threshold recipes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, average_precision_score, roc_auc_score, brier_score_loss
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
generator=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024,includeChirality=False)
X_fp=np.stack([generator.GetFingerprintAsNumPy(m) for m in molecules])
fingerprint_model=LogisticRegression(C=1.0,class_weight='balanced',max_iter=300,random_state=SEED)
fingerprint_model.fit(X_fp[train_id],y[train_id])
with torch.inference_mode():
    validation_scores={'PyG GINE':torch.sigmoid(forward_batch(model,full_batches['validation'])).numpy(),
                       'Morgan logistic':fingerprint_model.predict_proba(X_fp[val_id])[:,1]}
THRESHOLDS=np.array([0.2,0.35,0.5,0.65,0.8])
threshold_rows=[]
chosen_thresholds={}
for name,score in validation_scores.items():
    values=[balanced_accuracy_score(y[val_id],score>=t) for t in THRESHOLDS]
    chosen_thresholds[name]=float(THRESHOLDS[int(np.argmax(values))])
    threshold_rows.extend({'model':name,'threshold':float(t),'validation_balanced_accuracy':v}
                          for t,v in zip(THRESHOLDS,values))
threshold_table=pd.DataFrame(threshold_rows)
display(threshold_table)
print('Frozen thresholds:',chosen_thresholds)

## 12.8.6 Frozen test assessment

Read these measures together:

| Measure | Question |
| --- | --- |
| Sensitivity | Of positive labels, what fraction are called positive? |
| Specificity | Of negative labels, what fraction are called negative? |
| Balanced accuracy | How do those two class recalls average? |
| Average precision (AP) | How well are positives ranked across recall levels? Compare with prevalence. |
| ROC AUC | How often does a positive receive a higher score than a negative, accounting for ties? |
| Brier score | How close are the numerical scores to binary outcomes? Lower is better; it includes calibration effects. |

The confusion matrix exposes actual sample counts. AP is not interchangeable with trapezoidal area under a precision–recall curve. A constant training-prevalence baseline supplies a useful reference. Results describe the curated, small scaffold holdout; they do not establish clinical probabilities or universal GNN superiority.

In [ ]:
with torch.inference_mode():
    test_scores={'PyG GINE':torch.sigmoid(forward_batch(model,full_batches['test'])).numpy(),
                 'Morgan logistic':fingerprint_model.predict_proba(X_fp[test_id])[:,1],
                 'Training prior':np.full(len(test_id),y[train_id].mean())}
thresholds={**chosen_thresholds,'Training prior':0.5}
metric_rows=[]
for name,score in test_scores.items():
    predicted=score>=thresholds[name]
    tn,fp,fn,tp=confusion_matrix(y[test_id],predicted,labels=[0,1]).ravel()
    metric_rows.append({'model':name,'threshold':thresholds[name],
        'balanced_accuracy':balanced_accuracy_score(y[test_id],predicted),
        'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp),
        'AP':average_precision_score(y[test_id],score),'ROC_AUC':roc_auc_score(y[test_id],score),
        'Brier':brier_score_loss(y[test_id],score)})
metrics=pd.DataFrame(metric_rows).set_index('model')
display(metrics.round(3))
metrics.to_csv(OUT/'test_metrics.csv')
test_table=sample.iloc[test_id][['source_row','canonical','group','y']].copy()
for name,score in test_scores.items():test_table[name]=score
test_table.to_csv(OUT/'test_predictions.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(9.6,3.8),layout='constrained')
ConfusionMatrixDisplay.from_predictions(y[test_id],test_scores['PyG GINE']>=thresholds['PyG GINE'],
    labels=[0,1],display_labels=['Negative','Positive'],ax=axes[0],colorbar=False,cmap='Blues')
axes[0].set_title('PyG: threshold chosen on validation')
for name in ['PyG GINE','Morgan logistic']:
    precision,recall,_=precision_recall_curve(y[test_id],test_scores[name])
    axes[1].plot(recall,precision,label=name)
axes[1].axhline(y[test_id].mean(),color='black',linestyle=':',label='Test positive fraction')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Frozen test ranking',ylim=(0,1.03),xlim=(0,1))
axes[1].legend(fontsize=8)
fig.savefig(OUT/'classification_assessment.png',dpi=140)
plt.show()

## 12.8.7 A real PyG explanation of one model prediction

We select the first validation molecule by original row ID with 3–18 heavy atoms, solely to make a readable diagram. The choice uses neither its label nor prediction confidence. `GNNExplainer` fits soft node-feature and directed-edge masks to preserve the model's inferred class while regularizing the masks. The network weights stay fixed.

`explanation_type='model'` explains the **model's prediction**, including a possible wrong prediction. It does not explain the experimental phenomenon. The model configuration declares graph-level binary classification with raw logits. Node masks act on numerical features; edge masks act on messages. These masks are not probabilities that atoms or bonds cause permeability, and fractional features are not valid substituted molecules.

The 25-step budget demonstrates the real API without an expensive search; it does not guarantee converged or stable masks. We run two seeds, inspect their difference, and measure the masked score. [PyG explanation API](https://pytorch-geometric.readthedocs.io/en/2.8.0/modules/explain.html), [original GNNExplainer paper](https://arxiv.org/abs/1903.03894).

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer

eligible_probes=[int(i) for i in val_id if 3 <= molecules[i].GetNumHeavyAtoms() <= 18]
probe_id=min(eligible_probes,key=lambda i:int(sample.iloc[i].source_row))
probe=Batch.from_data_list([graphs[probe_id]])
probe_mol=molecules[probe_id]
model.eval()
before={k:v.detach().clone() for k,v in model.state_dict().items()}
with torch.inference_mode():original_logit=forward_batch(model,probe).item()
explanations=[]
masked_scores=[]
for explanation_seed in [1208,1209]:
    torch.manual_seed(explanation_seed)
    explainer=Explainer(model=model,algorithm=GNNExplainer(epochs=25,lr=0.01),
        explanation_type='model',node_mask_type='attributes',edge_mask_type='object',
        model_config={'mode':'binary_classification','task_level':'graph','return_type':'raw'})
    explanation=explainer(probe.x,probe.edge_index,edge_attr=probe.edge_attr,batch=probe.batch)
    assert explanation.node_mask.shape==probe.x.shape
    assert explanation.edge_mask.shape==(probe.num_edges,)
    assert torch.isfinite(explanation.node_mask).all() and torch.isfinite(explanation.edge_mask).all()
    masked_logit=explainer.get_masked_prediction(probe.x,probe.edge_index,
        node_mask=explanation.node_mask,edge_mask=explanation.edge_mask,
        edge_attr=probe.edge_attr,batch=probe.batch).item()
    masked_scores.append(float(torch.sigmoid(torch.tensor(masked_logit))))
    explanations.append(explanation)
for key,value in model.state_dict().items():torch.testing.assert_close(value,before[key])
with torch.inference_mode():assert np.isclose(forward_batch(model,probe).item(),original_logit)
print('Validation source row:',int(sample.iloc[probe_id].source_row))
print('Original sigmoid score (uncalibrated):',float(torch.sigmoid(torch.tensor(original_logit))))
print('Masked scores for two explanation seeds:',masked_scores)
print('Mean absolute directed-edge mask difference:',
      float((explanations[0].edge_mask-explanations[1].edge_mask).abs().mean()))

### Read the mask as a diagnostic, not a mechanism

Every chemical bond has two directed mask entries. The figure displays their **mean** as a visual summary; it does not force the optimizer to assign symmetric weights. Red intensity shows the raw 0–1 mask value. Check the masked score and seed disagreement alongside the drawing. A sparse attractive picture with poor retention of the original prediction is not automatically a faithful explanation.

A chemical hypothesis would require valid structural edits, recomputed features, a suitable experimental comparison, and independent evidence. The graph explainer supplies questions for that process, not a causal answer.

In [ ]:
edge_mask=explanations[0].edge_mask.detach().cpu()
pair_masks=edge_mask.reshape(-1,2).mean(1).numpy()
assert len(pair_masks)==probe_mol.GetNumBonds()
bond_colors={i:(1.0,1.0-0.8*float(v),1.0-0.8*float(v)) for i,v in enumerate(pair_masks)}
drawer=rdMolDraw2D.MolDraw2DCairo(680,330)
drawer.drawOptions().addAtomIndices=True
drawer.DrawMolecule(probe_mol,legend='GNNExplainer: mean directed-edge mask; not a causal map',
                    highlightAtoms=[],highlightBonds=list(bond_colors),highlightBondColors=bond_colors)
drawer.FinishDrawing()
png=drawer.GetDrawingText()
(OUT/'explained_validation_molecule.png').write_bytes(png)
display(Image(data=png))
fig,ax=plt.subplots(figsize=(8,3.3),layout='constrained')
labels=[f'{b.GetBeginAtomIdx()}-{b.GetEndAtomIdx()}' for b in probe_mol.GetBonds()]
for offset,explanation in zip([-0.18,0.18],explanations):
    values=explanation.edge_mask.detach().cpu().reshape(-1,2).mean(1).numpy()
    ax.bar(np.arange(len(labels))+offset,values,width=0.35,label=f'Seed {1208+int(offset>0)}')
ax.set_xticks(range(len(labels)),labels,rotation=60)
ax.set(xlabel='Bond endpoints',ylabel='Mean directed mask value',ylim=(0,1),title='Short explanations can depend on initialization')
ax.legend(fontsize=8)
fig.savefig(OUT/'explanation_seed_comparison.png',dpi=140)
plt.show()

## 12.8.8 Save the scientific protocol with the classifier

Inference needs the feature schema, architecture, weights, selected threshold, output convention and data/split provenance. Class weighting and threshold selection are part of the record. The explainer masks are stored separately: they do not change the classifier's weights or define a new predictive model.

In [ ]:
checkpoint={'state_dict':model.state_dict(),'architecture':ARCHITECTURE,'feature_schema':FEATURE_SCHEMA,
    'target':'BBBP reported binary p_np label','output':'raw logit; sigmoid score is not calibrated',
    'threshold':chosen_thresholds['PyG GINE'],'positive_weight':positive_weight,'versions':VERSIONS,
    'source_sha256':source_sha256,'best_epoch':best_epoch,'seed':SEED,
    'rows_by_partition':{p:sample.iloc[ids].source_row.tolist() for p,ids in indices.items()}}
torch.save(checkpoint,OUT/'pyg_bbbp.pt')
saved=torch.load(OUT/'pyg_bbbp.pt',map_location='cpu',weights_only=True)
restored=PropertyGINE(**saved['architecture']).eval()
restored.load_state_dict(saved['state_dict'])
with torch.inference_mode():torch.testing.assert_close(forward_batch(restored,probe),forward_batch(model,probe))
torch.save({'source_row':int(sample.iloc[probe_id].source_row),
    'node_masks':[e.node_mask.detach() for e in explanations],
    'edge_masks':[e.edge_mask.detach() for e in explanations]},OUT/'explanation_masks.pt')
record={'scope':'500-row curated teaching classification; no clinical or benchmark claim',
    'versions':VERSIONS,'source_sha256':source_sha256,'architecture':ARCHITECTURE,'feature_schema':FEATURE_SCHEMA,
    'seed':SEED,'split_counts':{p:len(ids) for p,ids in indices.items()},
    'training':{'optimizer':'Adam','learning_rate':0.005,'weight_decay':1e-4,'batch_size':64,
        'max_epochs':MAX_EPOCHS,'epochs_run':epoch,'best_epoch':best_epoch,'patience':PATIENCE,
        'min_delta':MIN_DELTA,'positive_weight':positive_weight,'seconds':training_seconds},
    'threshold_grid':THRESHOLDS.tolist(),'threshold_rule':'maximize validation balanced accuracy; lower threshold breaks ties',
    'selected_thresholds':chosen_thresholds,'test_metrics':metrics.to_dict(orient='index'),
    'explanation':{'type':'model','algorithm':'PyG GNNExplainer','epochs':25,'seeds':[1208,1209],
        'source_row':int(sample.iloc[probe_id].source_row),'masked_scores':masked_scores}}
(OUT/'experiment.json').write_text(json.dumps(record,indent=2)+'\n',encoding='utf-8')
print('Saved classifier, threshold, provenance, scores and explanation masks; checkpoint reload passed.')

## Exercises and answers

1. Why can accuracy look high when specificity is poor in this dataset?
2. Why derive class weights from training data and choose thresholds on validation data?
3. Why does a class-weighted sigmoid output need calibration assessment before probability interpretation?
4. What is being optimized by the explainer? Are the model weights changed?
5. What does averaging two directed bond masks lose? Why check more than one explanation seed?
6. Design a valid next research step after identifying a strongly highlighted bond.

<details><summary>Suggested answers</summary>

1. Positive labels are common. Predicting most molecules as positive can give high ordinary accuracy while missing many negatives. Report both class recalls and sample counts.
2. They are learned decisions. The final test should assess the whole frozen procedure rather than help define it.
3. Weighting changes the objective's preferred odds, and finite-data/model error adds further miscalibration. A useful ranking score need not be an empirical probability in the target population.
4. Soft masks on input features and messages, with a prediction-preservation objective and regularization. The classifier's parameters remain fixed; the notebook verifies that.
5. It hides directional disagreement. Initialization and a short optimization budget may change the masks, so examine stability and the masked prediction as well as appearance.
6. Formulate a specific, chemically valid structural comparison; check valence, state and assay context; recompute model features; specify independent evaluation or experiments. A mask alone does not prove that breaking or replacing that bond causes the endpoint.

</details>

**Primary references:** [GNNExplainer](https://arxiv.org/abs/1903.03894), [PyG explanation API](https://pytorch-geometric.readthedocs.io/en/2.8.0/modules/explain.html), [MoleculeNet](https://doi.org/10.1039/C7SC02664A), [calibration](https://proceedings.mlr.press/v70/guo17a.html), and [local dataset provenance](datasets/README.md).

[Back to 12.7](Chapter12_Part7.ipynb) · [Geometric learning in 12.5](Chapter12_Part5.ipynb) · [Course guide and research projects](docs/course-guide.md)